# Chemical Composition (PMDCo): from data entry to RDF

This notebook shows how to describe the chemical composition of a material
and convert that description into a standardised, machine-readable RDF graph
following the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.input.json
  │  your material name and element fractions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
```

---

## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # PMDCo/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))


Schema : schemas/chemical-composition/PMDCo


---
## Step 1: Describe your material

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `material_name` | yes | A name or identifier for the material |
| `elements` | yes | List of `{symbol, value, unit}`, one per element |
| `material_id` | no | Custom IRI slug (auto-derived from `material_name` if omitted) |
| `comp_id` | no | Custom IRI slug for the composition node (auto-derived if omitted) |

Element `unit` must be `"mass%"`, `"vol%"`, or `"mol%"`.  All elements in one
composition should use the same unit.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "material_name": "316L Stainless Steel",
  "elements": [
    {
      "symbol": "Fe",
      "value": 65.345,
      "unit": "mass%"
    },
    {
      "symbol": "Cr",
      "value": 17.0,
      "unit": "mass%"
    },
    {
      "symbol": "Ni",
      "value": 12.0,
      "unit": "mass%"
    },
    {
      "symbol": "Mo",
      "value": 2.5,
      "unit": "mass%"
    },
    {
      "symbol": "Mn",
      "value": 1.8,
      "unit": "mass%"
    },
    {
      "symbol": "Si",
      "value": 0.5,
      "unit": "mass%"
    },
    {
      "symbol": "N",
      "value": 0.1,
      "unit": "mass%"
    },
    {
      "symbol": "C",
      "value": 0.03,
      "unit": "mass%"
    },
    {
      "symbol": "P",
      "value": 0.045,
      "unit": "mass%"
    },
    {
      "symbol": "S",
      "value": 0.03,
      "unit": "mass%"
    }
  ]
}


---
## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document, the intermediate
format that carries both the data and its ontology mapping.

Node identifiers (`material_id`, `comp_id`) are derived automatically from
`material_name` as a URL-safe slug (e.g. `"316L Stainless Steel"` →
`mat-316l-stainless-steel`).  Override them in the input only when you need
a specific IRI to match an existing knowledge graph node.

You can also run the transform from the command line:
```
python -m jsonata -e ../specs/transform.simplified.jsonata -i example.input.json
```

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/PMDCo/#v1.0.0",
  "type": "pmdco:PMD_0000551",
  "id": "chem-comp-316l-stainless-steel",
  "quality_of": {
    "label": "316L Stainless Steel"
  },
  "is_subject_of": {
    "type": "pmdco:PMD_0025002",
    "id": "chem-comp-316l-stainless-steel-spec",
    "has_member": [
      {
        "type": "pmdco:PMD_0025997",
        "id": "frac-Fe",
        "value": 65.345,
        "unit": "uo:0000163",
        "element": {
          "type": "pmdco:PMD_0020026",
          "id": "elem-Fe",
          "part_of": "mat-316l-stainless-steel",
          "has_relational_quality": {
            "type": "pmdco:PMD_0020102",
            "id": "prop-Fe",
            "relational_quality_of": "elem-Fe",
            "specified_by_value": "frac-Fe"
          }
        }
      },
      {
        "type": "pmdco:PMD_0025997",
        "id": "frac-Cr",
        "value": 17.0,
        "unit": "uo:0000163",


---
## Step 3: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [5]:
flat = schema.to_graph(simplified)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 116 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/chem-comp-316l-stainless-steel> a pmdco:PMD_0000551 ;
    obo:RO_0000080 [ rdfs:label "316L Stainless Steel" ] ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/PMDCo/#v1.0.0> ;
    pmdco:PMD_0000004 <https://w3id.org/pmd/co/test/chem-comp-316l-stainless-steel-spec> .

<https://w3id.org/pmd/co/test/chem-comp-316l-stainless-steel-spec> a pmdco:PMD_0025002 ;
    obo:RO_0002351 <https://w3id.org/pmd/co/test/frac-C>,
        <https://w3id.org/pmd/co/test/frac-Cr>,
        <https://w3id.org/pmd/co/test/frac-Fe>,
        <https://w3id.org/pmd/co/test/frac-Mn>,
        <https://w3id.org/pmd/co/test/frac-Mo>,
        <https

In [6]:
# Optional: save to file
out_ttl = HERE / "output_316L.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_316L.ttl


---
## Step 4: Validate against the SHACL shape

The SHACL shape (`specs/shape.ttl`) checks that the graph is structurally
correct; for example, that every element fraction has a numeric value
in \[0, 100\] and an allowed unit IRI.

> `inference="rdfs"` is required because some shapes target superclasses
> (e.g. `Proportion`, `PortionOfSingleChemicalElement`) and rely on RDFS
> reasoning to match the specific subtypes used in the data.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


---
## Step 5: Inspect the graph

The cell below uses SPARQL to retrieve all element fractions from the graph
and display them in a table, sorted by value (highest first).

You do not need to understand SPARQL to use this notebook. Just run the cell
and check that the output matches what you entered.

In [8]:
SPARQL_COMP = """
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX ro:   <http://purl.obolibrary.org/obo/RO_>
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:  <http://purl.obolibrary.org/obo/IAO_>

SELECT ?symbol ?value ?unit
WHERE {
  ?comp a pmd:PMD_0000551 ;
        pmd:PMD_0000004 ?spec .
  ?spec ro:0002351 ?frac .
  ?frac obi:0001937 ?value ;
        iao:0000039 ?unit_iri ;
        pmd:PMD_hasFractionElement ?elem .
  BIND(REPLACE(STR(?elem), ".*/elem-", "") AS ?symbol)
  BIND(REPLACE(STR(?unit_iri), ".*/", "") AS ?unit)
}
ORDER BY DESC(?value)
"""

rows = list(flat.query(SPARQL_COMP))
if rows:
    print(f"  {'Element':<8}  {'Value':>10}  Unit")
    print("  " + "-" * 30)
    for r in rows:
        val = f"{float(r.value):>10.4g}"
        print(f"  {str(r.symbol):<8}  {val}  {r.unit}")
else:
    print("No element fractions found.")

  Element        Value  Unit
  ------------------------------
  Fe             65.34  UO_0000163
  Cr                17  UO_0000163
  Ni                12  UO_0000163
  Mo               2.5  UO_0000163
  Mn               1.8  UO_0000163
  Si               0.5  UO_0000163
  N                0.1  UO_0000163
  P              0.045  UO_0000163
  S               0.03  UO_0000163
  C               0.03  UO_0000163


---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the material name and element fractions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Element fractions are extracted from the graph to confirm correctness |

To use your own material, edit `docs/example.input.json` and re-run all cells.

---

## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/3_schema-format.md): how to write your own schema
- [Simplified input guide](../../../docs/simplified-input-guide.md): end-to-end workflow